# 🚗 Reconhecimento de Placas de Veículos

Este notebook realiza a identificação e leitura de placas de veículos em imagens utilizando:
- **OpenCV**: Para processamento de imagem e detecção da região da placa
- **EasyOCR**: Para reconhecimento óptico de caracteres (OCR)

---

## 1. Instalação das Dependências

In [ ]:
# Instalando as bibliotecas necessárias
!pip install easyocr opencv-python-headless matplotlib imutils -q
print("Dependências instaladas com sucesso!")

## 2. Importação das Bibliotecas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import easyocr
import imutils
from google.colab import files
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas importadas com sucesso!")

## 3. Funções Auxiliares

In [ ]:
def exibir_imagem(imagem, titulo="Imagem", cmap=None):
    """
    Exibe uma imagem usando matplotlib.
    
    Args:
        imagem: Imagem a ser exibida (BGR ou escala de cinza)
        titulo: Título da imagem
        cmap: Mapa de cores (usar 'gray' para escala de cinza)
    """
    plt.figure(figsize=(12, 8))
    if len(imagem.shape) == 3:
        # Converter de BGR para RGB para exibição correta
        plt.imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(imagem, cmap='gray')
    plt.title(titulo, fontsize=14)
    plt.axis('off')
    plt.show()


def preprocessar_imagem(imagem):
    """
    Realiza o pré-processamento da imagem para detecção de bordas.
    
    Args:
        imagem: Imagem original em BGR
        
    Returns:
        tuple: (imagem em escala de cinza, imagem com bordas detectadas)
    """
    # Converter para escala de cinza
    cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    
    # Aplicar filtro bilateral para redução de ruído mantendo bordas
    filtrada = cv2.bilateralFilter(cinza, 11, 17, 17)
    
    # Detectar bordas usando Canny
    bordas = cv2.Canny(filtrada, 30, 200)
    
    return cinza, bordas


def encontrar_contorno_placa(bordas):
    """
    Encontra o contorno que provavelmente representa a placa do veículo.
    
    Args:
        bordas: Imagem com bordas detectadas
        
    Returns:
        numpy.ndarray ou None: Contorno da placa (4 pontos) ou None se não encontrado
    """
    # Encontrar contornos
    contornos = cv2.findContours(bordas.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    contornos = imutils.grab_contours(contornos)
    
    # Ordenar contornos por área (maior para menor)
    contornos = sorted(contornos, key=cv2.contourArea, reverse=True)[:10]
    
    contorno_placa = None
    
    for contorno in contornos:
        # Aproximar o contorno para um polígono
        perimetro = cv2.arcLength(contorno, True)
        aproximacao = cv2.approxPolyDP(contorno, 0.018 * perimetro, True)
        
        # Se o polígono tem 4 vértices, provavelmente é a placa
        if len(aproximacao) == 4:
            contorno_placa = aproximacao
            break
    
    return contorno_placa


def extrair_regiao_placa(imagem, contorno):
    """
    Extrai a região da placa da imagem original.
    
    Args:
        imagem: Imagem original
        contorno: Contorno da placa (4 pontos)
        
    Returns:
        numpy.ndarray: Região da placa recortada
    """
    # Criar máscara
    mascara = np.zeros(imagem.shape[:2], np.uint8)
    cv2.drawContours(mascara, [contorno], 0, 255, -1)
    
    # Aplicar máscara na imagem
    imagem_mascarada = cv2.bitwise_and(imagem, imagem, mask=mascara)
    
    # Obter bounding box do contorno
    x, y, w, h = cv2.boundingRect(contorno)
    
    # Recortar região da placa
    placa_recortada = imagem[y:y+h, x:x+w]
    
    return placa_recortada


def ler_texto_placa(imagem_placa, reader):
    """
    Realiza OCR na imagem da placa para extrair o texto.
    
    Args:
        imagem_placa: Imagem da região da placa
        reader: Instância do EasyOCR Reader
        
    Returns:
        str: Texto lido da placa
    """
    # Realizar OCR
    resultados = reader.readtext(imagem_placa)
    
    # Concatenar todos os textos encontrados
    texto_placa = ""
    for (bbox, texto, prob) in resultados:
        texto_placa += texto + " "
    
    return texto_placa.strip()


print("Funções auxiliares carregadas com sucesso!")

## 4. Inicialização do Leitor OCR

O EasyOCR será configurado para reconhecer textos em português e inglês.

In [ ]:
# Inicializar o leitor EasyOCR
# Suporta português (pt) e inglês (en) para melhor reconhecimento de placas
print("Inicializando o leitor OCR (isso pode levar alguns segundos na primeira execução)...")
reader = easyocr.Reader(['pt', 'en'], gpu=True)
print("Leitor OCR inicializado com sucesso!")

## 5. Upload da Imagem

Faça upload de uma imagem contendo um veículo com placa visível.

In [ ]:
# Upload da imagem
print("Por favor, faça upload de uma imagem contendo uma placa de veículo:")
uploaded = files.upload()

# Obter o nome do arquivo enviado
nome_arquivo = list(uploaded.keys())[0]
print(f"\nArquivo carregado: {nome_arquivo}")

## 6. Processamento da Imagem e Detecção da Placa

In [ ]:
# Carregar a imagem
imagem_original = cv2.imread(nome_arquivo)

# Verificar se a imagem foi carregada corretamente
if imagem_original is None:
    raise ValueError("Erro ao carregar a imagem. Verifique o arquivo.")

# Redimensionar se necessário (para melhor processamento)
imagem = imutils.resize(imagem_original, width=620)

print(f"Dimensões da imagem: {imagem.shape[1]}x{imagem.shape[0]} pixels")

# Exibir imagem original
exibir_imagem(imagem, "Imagem Original")

In [ ]:
# Pré-processar a imagem
cinza, bordas = preprocessar_imagem(imagem)

# Exibir resultados do pré-processamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(cinza, cmap='gray')
axes[0].set_title('Imagem em Escala de Cinza', fontsize=12)
axes[0].axis('off')

axes[1].imshow(bordas, cmap='gray')
axes[1].set_title('Detecção de Bordas (Canny)', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Encontrar o contorno da placa
contorno_placa = encontrar_contorno_placa(bordas)

if contorno_placa is not None:
    print("Placa detectada com sucesso!")
    
    # Desenhar contorno na imagem
    imagem_com_contorno = imagem.copy()
    cv2.drawContours(imagem_com_contorno, [contorno_placa], -1, (0, 255, 0), 3)
    
    exibir_imagem(imagem_com_contorno, "Placa Detectada")
else:
    print("Não foi possível detectar a placa automaticamente.")
    print("Tentando método alternativo com OCR direto na imagem...")

## 7. Extração e Leitura da Placa

In [ ]:
if contorno_placa is not None:
    # Extrair região da placa
    placa_recortada = extrair_regiao_placa(imagem, contorno_placa)
    
    # Exibir placa recortada
    exibir_imagem(placa_recortada, "Região da Placa Extraída")
    
    # Ler texto da placa
    texto_placa = ler_texto_placa(placa_recortada, reader)
else:
    # Se não detectou contorno, tentar OCR na imagem inteira
    print("Aplicando OCR na imagem completa...")
    texto_placa = ler_texto_placa(imagem, reader)

print("\n" + "="*50)
print("RESULTADO DA LEITURA DA PLACA")
print("="*50)
print(f"\nTexto identificado: {texto_placa}")
print("\n" + "="*50)

## 8. Resultado Final com Anotações

In [ ]:
# Criar imagem final com anotações
imagem_final = imagem.copy()

if contorno_placa is not None:
    # Desenhar contorno da placa
    cv2.drawContours(imagem_final, [contorno_placa], -1, (0, 255, 0), 3)
    
    # Obter posição para o texto
    x, y, w, h = cv2.boundingRect(contorno_placa)
    
    # Adicionar fundo para o texto
    cv2.rectangle(imagem_final, (x, y - 40), (x + w, y), (0, 255, 0), -1)
    
    # Adicionar texto da placa
    cv2.putText(imagem_final, texto_placa, (x + 5, y - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

# Exibir resultado final
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(imagem_final, cv2.COLOR_BGR2RGB))
plt.title(f'Resultado Final - Placa: {texto_placa}', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

# Salvar imagem resultado
cv2.imwrite('resultado_placa.jpg', imagem_final)
print(f"\nImagem com resultado salva como: resultado_placa.jpg")

## 9. Método Alternativo: Detecção com OCR Avançado

Este método utiliza o EasyOCR diretamente para detectar e ler textos, útil quando a detecção por contornos falha.

In [ ]:
def detectar_placa_ocr_avancado(imagem, reader):
    """
    Método alternativo que usa OCR diretamente para detectar texto.
    Filtra resultados para encontrar padrões de placa.
    
    Args:
        imagem: Imagem do veículo
        reader: Instância do EasyOCR Reader
        
    Returns:
        list: Lista de resultados OCR com bounding boxes
    """
    # Realizar OCR com detecção de parágrafo
    resultados = reader.readtext(imagem, paragraph=False)
    
    # Filtrar resultados com alta confiança
    resultados_filtrados = [(bbox, texto, prob) for bbox, texto, prob in resultados if prob > 0.3]
    
    return resultados_filtrados


# Aplicar método alternativo
print("Aplicando detecção OCR avançada...")
resultados_ocr = detectar_placa_ocr_avancado(imagem, reader)

# Criar imagem com todas as detecções
imagem_deteccoes = imagem.copy()

print(f"\nForam encontrados {len(resultados_ocr)} textos na imagem:\n")

for i, (bbox, texto, prob) in enumerate(resultados_ocr):
    # Converter pontos para inteiros
    pontos = np.array(bbox, dtype=np.int32)
    
    # Desenhar polígono ao redor do texto
    cv2.polylines(imagem_deteccoes, [pontos], True, (255, 0, 0), 2)
    
    # Adicionar texto e probabilidade
    x_min = int(min([p[0] for p in bbox]))
    y_min = int(min([p[1] for p in bbox]))
    
    cv2.putText(imagem_deteccoes, texto, (x_min, y_min - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    print(f"  {i+1}. Texto: '{texto}' (Confiança: {prob:.2%})")

# Exibir todas as detecções
exibir_imagem(imagem_deteccoes, "Todas as Detecções de Texto")

## 10. Identificação de Padrão de Placa Brasileira

Filtra os resultados para encontrar padrões típicos de placas brasileiras:
- **Padrão Antigo**: 3 letras + 4 números (ABC-1234)
- **Padrão Mercosul**: 3 letras + 1 número + 1 letra + 2 números (ABC1D23)

In [ ]:
import re

def identificar_placa_brasileira(texto):
    """
    Verifica se o texto corresponde ao padrão de placa brasileira.
    
    Args:
        texto: Texto a ser verificado
        
    Returns:
        tuple: (bool, str) - Se é placa e o padrão identificado
    """
    # Remover espaços e caracteres especiais
    texto_limpo = re.sub(r'[^A-Za-z0-9]', '', texto.upper())
    
    # Padrão antigo: 3 letras + 4 números
    padrao_antigo = r'^[A-Z]{3}[0-9]{4}$'
    
    # Padrão Mercosul: 3 letras + 1 número + 1 letra + 2 números
    padrao_mercosul = r'^[A-Z]{3}[0-9][A-Z][0-9]{2}$'
    
    if re.match(padrao_antigo, texto_limpo):
        return True, "Padrão Antigo", texto_limpo
    elif re.match(padrao_mercosul, texto_limpo):
        return True, "Padrão Mercosul", texto_limpo
    else:
        return False, None, texto_limpo


# Verificar se algum texto detectado corresponde a uma placa
print("Verificando padrões de placa brasileira...\n")

placas_encontradas = []

for bbox, texto, prob in resultados_ocr:
    eh_placa, padrao, texto_limpo = identificar_placa_brasileira(texto)
    
    if eh_placa:
        placas_encontradas.append({
            'texto': texto_limpo,
            'padrao': padrao,
            'confianca': prob,
            'bbox': bbox
        })
        print(f"PLACA ENCONTRADA: {texto_limpo}")
        print(f"  - Padrão: {padrao}")
        print(f"  - Confiança: {prob:.2%}\n")

if not placas_encontradas:
    print("Nenhum padrão de placa brasileira identificado nos textos detectados.")
    print("Exibindo texto mais provável como placa:")
    if resultados_ocr:
        melhor_resultado = max(resultados_ocr, key=lambda x: x[2])
        print(f"  - Texto: {melhor_resultado[1]}")
        print(f"  - Confiança: {melhor_resultado[2]:.2%}")

## 11. Resumo e Download do Resultado

In [ ]:
# Resumo final
print("\n" + "="*60)
print("                    RESUMO FINAL")
print("="*60)

if placas_encontradas:
    for i, placa in enumerate(placas_encontradas, 1):
        print(f"\nPlaca {i}:")
        print(f"  Texto: {placa['texto']}")
        print(f"  Padrão: {placa['padrao']}")
        print(f"  Confiança: {placa['confianca']:.2%}")
else:
    print(f"\nTexto identificado na placa: {texto_placa}")

print("\n" + "="*60)

# Download do resultado
print("\nBaixando imagem com resultado...")
files.download('resultado_placa.jpg')

---

## Observações

### Dicas para melhores resultados:
1. **Qualidade da imagem**: Use imagens com boa resolução e iluminação adequada
2. **Ângulo**: Placas fotografadas de frente têm melhor reconhecimento
3. **Contraste**: Imagens com bom contraste entre a placa e o fundo funcionam melhor

### Limitações:
- Placas muito inclinadas ou com reflexos podem não ser detectadas corretamente
- Imagens de baixa qualidade podem resultar em leituras incorretas
- O algoritmo foi otimizado para placas brasileiras

### Tecnologias utilizadas:
- **OpenCV**: Processamento de imagem e detecção de contornos
- **EasyOCR**: Reconhecimento óptico de caracteres com suporte a múltiplos idiomas
- **imutils**: Funções auxiliares para manipulação de imagens